In [0]:
import mlflow

In [0]:
mlflow.set_experiment("/Users/deepak.sarkar@mindsprint.com/Evaluation-Quickstart")

In [0]:
from openai import OpenAI
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=token,
    base_url=f"{w.config.host}/serving-endpoints",
)


def my_agent(question: str) -> str:
    response = client.chat.completions.create(
        model="databricks-meta-llama-3-3-70b-instruct",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. Answer questions concisely.",
            },
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content


def qa_predict_fn(question: str) -> str:
    """Wrapper function for evaluation using ``my_agent``."""
    return my_agent(question)

In [0]:
# Define a simple Q&A dataset with questions and expected answers
eval_dataset = [
    {
        "inputs": {"question": "What is the capital of France?"},
        "expectations": {"expected_response": "Paris"},
    },
    {
        "inputs": {"question": "Who was the first person to build an airplane?"},
        "expectations": {"expected_response": "Wright Brothers"},
    },
    {
        "inputs": {"question": "Who wrote Romeo and Juliet?"},
        "expectations": {"expected_response": "William Shakespeare"},
    },
]

In [0]:
from mlflow.genai import scorer
from mlflow.genai.scorers import Correctness, Guidelines


@scorer
def is_concise(outputs: str) -> bool:
    """Evaluate if the answer is concise (less than 5 words)"""
    return len(outputs.split()) <= 5


scorers = [
    Correctness(),
    Guidelines(name="is_spanish", guidelines="The answer must be in Spanish"),
    is_concise,
]

In [0]:
# Run evaluation
if __name__ == "__main__":
    results = mlflow.genai.evaluate(
        data=eval_dataset,
        predict_fn=qa_predict_fn,
        scorers=scorers,
    )